# M0 walkthrough: one episode through ArbiterKB
Observations → KB (beliefs + evidence) → plan → score. Synthetic RoboMME-like data, CPU only (DEVELOPMENT.md §6 M0).

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.environ['ARBITER_DEBUG'] = '1'
from arbiter_kb.sources.synthetic import SyntheticSource, COVER_T
from arbiter_kb.sources.corrupt import CorruptedSource
from arbiter_kb.systems.arbiter_kb import ArbiterKBSystem
from arbiter_kb.views import text_view, ask
from arbiter_kb.debug import explain

**Perception stand-in.** A Source yields one `FrameObs` per frame: per object visible / xyz (m) / uv (px).

In [ ]:
src = SyntheticSource()
ep = 1025
meta, gt = src.meta(ep), src.ground_truth(ep)
frames = list(src.frames(ep))
print(meta.task_goal, '| frames:', len(frames))
print('truth:', gt.cube_to_bin, '| t*:', gt.t_star)
frames[COVER_T].objects[gt.targets[0]]

**Fast loop + KB.** Step frame by frame; print the KB's text view (what the arbiter will read) just after the cover, mid-run and at t*.

In [ ]:
s = ArbiterKBSystem()
s.reset(meta)
show = {COVER_T, 120, len(frames) - 1}
for f in frames:
    s.step(f)
    if f.t in show:
        print(f'--- t={f.t}')
        print(text_view(s.kb))

**Evidence log** (appear / disappear / move / near_pass), numbered so beliefs can cite it.

In [ ]:
for e in s.kb.evidence:
    if e.kind in ('disappear', 'near_pass'):
        print(e.id, e.t, e.kind, e.args, {k: v for k, v in e.data.items() if k != 'obs'})

**Predicate view** with the course's unify / ASK — answers carry probabilities.

In [ ]:
cube = gt.targets[0]
ask(s.kb, ('Inside', cube, '?c'))

**Planner → skill → 4 waypoints** `[x, y, z, roll, pitch, yaw, gripper]` from the KB's L0 only.

In [ ]:
step = s.plan(cube)
print(step.skill, step.target, round(step.conf, 3), step.decision)
step.waypoints.round(3)

**Why does it believe that?** The support chain down to raw observations (E2).

In [ ]:
print(explain(s.trace_rows(), cube))

**Same episode with a tracker identity swap.** Labels lie after the closest approach; the bound rule follows the wrong label with confidence 1, ArbiterKB keeps the alternative.

In [ ]:
bad = CorruptedSource(src, p=1.0)
from arbiter_kb.evaluate import run
from arbiter_kb.params import Params
from arbiter_kb.systems.baselines import TrackerSystem
rows, traces, _ = run([TrackerSystem('bound'), ArbiterKBSystem()], bad, [ep], Params(), 'id_swap')
print(bad.log[ep])
[(r['system'], r['pred'], r['truth'], r['p_max'], r['correct']) for r in rows]

**Summary figure** from the full M0 run (`make selftest` writes it).

In [ ]:
from IPython.display import Image
Image('../figures/m0_belief_timeline.png')